# Sistema de Puntos McDonald's — BATCH (Sistema Actual)

Todos los clientes ganan puntos a la misma tasa: **10 puntos por cada $1 gastado**. Los puntos se acreditan una vez al día en un proceso nocturno (batch).

---

## 1. Librerías y parámetros

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

np.random.seed(42)

N_CLIENTES    = 120
N_TRANSAC     = 600
INICIO        = datetime(2024, 4, 1)
FIN           = datetime(2024, 4, 30, 23, 59)
PUNTOS_X_PESO = 10   # tasa fija para todos los clientes
BATCH_HORAS   = 24

MC_ROJO    = '#DA291C'
MC_AMARILLO = '#FFC72C'
MC_GRIS    = '#27251F'
MC_CLARO   = '#F5F0E8'

## 2. Datos sintéticos

In [ ]:
# Clientes: todos con la misma tasa de puntos
clientes = pd.DataFrame({
    'cliente_id': range(1, N_CLIENTES + 1),
    'nombre'    : [f'Cliente_{i:03d}' for i in range(1, N_CLIENTES + 1)],
})

# Transacciones distribuidas aleatoriamente durante abril 2024
seg_totales = int((FIN - INICIO).total_seconds())

transacciones = pd.DataFrame({
    'tx_id'     : range(1, N_TRANSAC + 1),
    'cliente_id': np.random.choice(clientes['cliente_id'], size=N_TRANSAC),
    'monto'     : np.round(np.random.exponential(scale=8, size=N_TRANSAC) + 2, 2),
    'timestamp' : [
        INICIO + timedelta(seconds=int(s))
        for s in np.random.randint(0, seg_totales, N_TRANSAC)
    ],
})

transacciones = transacciones.sort_values('timestamp').reset_index(drop=True)
transacciones['puntos'] = np.floor(transacciones['monto'] * PUNTOS_X_PESO).astype(int)

print(f'Clientes: {len(clientes)} | Transacciones: {len(transacciones)}')
print(transacciones.head())

## 3. Limpieza de datos

In [ ]:
n_antes = len(transacciones)

transacciones = transacciones[transacciones['monto'] > 0]
transacciones = transacciones[
    (transacciones['timestamp'] >= INICIO) &
    (transacciones['timestamp'] <= FIN)
]
transacciones = transacciones.drop_duplicates(subset='tx_id').reset_index(drop=True)

print(f'Registros antes: {n_antes} | Después: {len(transacciones)} | Eliminados: {n_antes - len(transacciones)}')

## 4. Sistema BATCH

Los puntos de cada compra quedan disponibles al final del ciclo de 24 horas en que ocurrió la transacción. Si compras a las 8 AM, tus puntos aparecen a medianoche.

In [ ]:
# Ciclo batch: qué día nocturno procesa cada transacción
transacciones['ciclo'] = transacciones['timestamp'].apply(
    lambda ts: int((ts - INICIO).total_seconds() / 3600 // BATCH_HORAS)
)

# Momento en que el cliente puede ver los puntos
transacciones['disponible'] = transacciones['ciclo'].apply(
    lambda c: INICIO + timedelta(hours=(c + 1) * BATCH_HORAS)
)

# Latencia = tiempo de espera entre la compra y la disponibilidad
transacciones['latencia_h'] = (
    transacciones['disponible'] - transacciones['timestamp']
).dt.total_seconds() / 3600

print(f"Latencia media : {transacciones['latencia_h'].mean():.1f} h")
print(f"Latencia máxima: {transacciones['latencia_h'].max():.1f} h")
print(f"Latencia mínima: {transacciones['latencia_h'].min():.2f} h")

## 5. Visualización

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor(MC_CLARO)
fig.suptitle('Sistema BATCH — Puntos McDonald\'s', fontweight='bold', color=MC_GRIS)

# Gráfica 1: distribución de latencia
ax1 = axes[0]
ax1.set_facecolor(MC_CLARO)
ax1.hist(transacciones['latencia_h'], bins=24, color=MC_ROJO, edgecolor='white', alpha=0.85)
ax1.axvline(transacciones['latencia_h'].mean(), color=MC_GRIS, linestyle='--',
            label=f"Media: {transacciones['latencia_h'].mean():.1f} h")
ax1.set_title('Distribución de latencia', color=MC_GRIS, fontweight='bold')
ax1.set_xlabel('Horas de espera para ver los puntos')
ax1.set_ylabel('Transacciones')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)
ax1.spines[['top', 'right']].set_visible(False)

# Gráfica 2: top 10 clientes
ax2 = axes[1]
ax2.set_facecolor(MC_CLARO)
top10 = (
    transacciones.groupby('cliente_id')['puntos'].sum()
    .nlargest(10)
    .reset_index()
)
ax2.barh(
    top10['cliente_id'].astype(str), top10['puntos'],
    color=MC_ROJO, edgecolor=MC_GRIS, linewidth=0.5, alpha=0.85
)
ax2.set_title('Top 10 clientes por puntos', color=MC_GRIS, fontweight='bold')
ax2.set_xlabel('Puntos acumulados')
ax2.set_ylabel('Cliente ID')
ax2.invert_yaxis()
ax2.grid(axis='x', alpha=0.3)
ax2.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('batch_resultados.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. Resumen

In [ ]:
print('=== SISTEMA BATCH ===')
print(f"Transacciones : {len(transacciones)}")
print(f"Clientes activos: {transacciones['cliente_id'].nunique()}")
print(f"Puntos emitidos : {transacciones['puntos'].sum():,}")
print(f"Latencia media  : {transacciones['latencia_h'].mean():.1f} h")
print(f"Latencia máxima : {transacciones['latencia_h'].max():.1f} h")